# 06 — LTI Systems and Convolution

Notebooks 01–05 covered the *tools*. This one covers the **central theorem of the course**,
and it is worth being blunt about why it matters:

> If a system is linear and time-invariant, then knowing its response to a **single impulse**
> tells you its response to **every possible input**.

That is an extraordinary claim. There are infinitely many input signals; you measure one
response and you can predict all of them. The operation that turns "response to an impulse"
into "response to anything" is **convolution**.

Almost everything else in Signals and Systems is a consequence of this: transfer functions,
frequency response, filter design, stability criteria, the Laplace and z-transforms. They are
all tools for working with convolution more conveniently.

This notebook builds the result from scratch, verifies every step numerically, and then shows
what breaks when the assumptions fail.

**Prerequisites:** notebook 01 (arrays, `np.convolve`, FFT) and notebook 02 (`stem` plots).
Notebook 03 is helpful but not required.

---

## Contents

| § | Topic |
|---|-------|
| 1 | A system is an operator |
| 2 | Linearity, tested numerically |
| 3 | Time invariance, tested numerically |
| 4 | The impulse as a basis |
| 5 | Deriving convolution |
| 6 | The black-box experiment |
| 7 | Flip-and-slide, step by step |
| 8 | Convolution modes and support |
| 9 | Properties of convolution |
| 10 | Cascade and parallel systems |
| 11 | Continuous-time convolution |
| 12 | Causality, memory, and stability |
| 13 | Step response ↔ impulse response |
| 14 | Complex exponentials are eigenfunctions |
| 15 | The convolution theorem |
| 16 | Circular convolution and the zero-padding trap |
| 17 | Block convolution: overlap-add |
| 18 | Deconvolution, and why it is hard |
| 19 | Applications: smoothing, differencing, reverb, images |
| 20 | Common mistakes |
| 21 | Exercises |

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

np.set_printoptions(precision=4, suppress=True, linewidth=100)
plt.rcParams.update({
    "figure.figsize": (10, 3.0), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

def stem_ax(ax, n, x, color="tab:blue", label=None, ms=5):
    """Consistent discrete-signal plotting for the whole notebook."""
    ml, sl, bl = ax.stem(n, x, basefmt=" ", label=label)
    plt.setp(ml, markersize=ms, color=color)
    plt.setp(sl, linewidth=1.2, color=color, alpha=0.75)
    ax.axhline(0, color="k", lw=0.7)
    return ax

print("ready")

---
## 1. A system is an operator

A system $T$ maps an input signal to an output signal: $y = T\{x\}$. Note carefully that the
argument is the *whole signal*, not one sample. In Python a system is naturally a function
that takes an array and returns an array.

Here are six systems we will interrogate. Three are LTI and three are not — but you cannot
tell which by looking at them, which is the point of the tests in §2 and §3.

In [ ]:
def sys_scale(x):
    """y[n] = 3 x[n]  -- amplifier."""
    return 3.0 * x

def sys_movavg(x):
    """y[n] = (x[n] + x[n-1] + x[n-2]) / 3  -- 3-point moving average."""
    return np.convolve(x, np.ones(3) / 3, mode="full")[:len(x)]

def sys_leaky(x):
    """y[n] = 0.8 y[n-1] + x[n]  -- leaky integrator (recursive)."""
    return signal.lfilter([1.0], [1.0, -0.8], x)

def sys_square(x):
    """y[n] = x[n]^2  -- squarer."""
    return x ** 2

def sys_ramp_gain(x):
    """y[n] = n x[n]  -- gain that grows with time."""
    return np.arange(len(x)) * x

def sys_bias(x):
    """y[n] = x[n] + 2  -- adds a constant offset."""
    return x + 2.0

SYSTEMS = {
    "3x[n]": sys_scale,
    "moving average": sys_movavg,
    "leaky integrator": sys_leaky,
    "x[n]^2": sys_square,
    "n*x[n]": sys_ramp_gain,
    "x[n] + 2": sys_bias,
}

for name, f in SYSTEMS.items():
    print(f"{name:<20} {f.__doc__.splitlines()[0]}")

> **`sys_bias` is the interesting one.** Adding a constant feels harmless, and the system is
> perfectly time-invariant. It is *not linear* — and that single failure is enough to make
> every tool in this course inapplicable to it. Systems like this are called *affine*, and the
> standard trick is to analyse the deviation from the operating point instead.

---
## 2. Linearity, tested numerically

A system is **linear** if it satisfies superposition, which is two conditions in one:

- **Additivity:** $T\{x_1 + x_2\} = T\{x_1\} + T\{x_2\}$
- **Homogeneity (scaling):** $T\{a\,x\} = a\,T\{x\}$

Combined into a single test:

$$T\{a\,x_1 + b\,x_2\} \stackrel{?}{=} a\,T\{x_1\} + b\,T\{x_2\}$$

Rather than reason about it, just check it with random signals and random coefficients. If it
fails for one random draw, the system is not linear. (Passing many random draws is strong
evidence, though not a proof — a proof needs algebra.)

In [ ]:
def is_linear(system, n=64, trials=20, tol=1e-9, rng=None):
    """Test T{a x1 + b x2} == a T{x1} + b T{x2} for random x1, x2, a, b."""
    rng = np.random.default_rng(0) if rng is None else rng
    worst = 0.0
    for _ in range(trials):
        x1, x2 = rng.normal(size=n), rng.normal(size=n)
        a, b = rng.normal(), rng.normal()
        lhs = system(a * x1 + b * x2)
        rhs = a * system(x1) + b * system(x2)
        worst = max(worst, np.max(np.abs(lhs - rhs)))
    return worst < tol, worst

print(f"{'system':<20} {'linear?':<10} worst error")
print("-" * 46)
for name, f in SYSTEMS.items():
    ok, err = is_linear(f)
    print(f"{name:<20} {str(ok):<10} {err:.3e}")

Two fail, and they fail by different mechanisms:

- **`x[n]^2`** breaks homogeneity badly: doubling the input quadruples the output.
- **`x[n] + 2`** breaks additivity: the constant gets added twice on the right-hand side but
  only once on the left.

**`n*x[n]` passes.** It is genuinely linear — and still not LTI, because it is time-varying.
Linearity alone buys you nothing; §3 is the other half of the test.

In [ ]:
n = np.arange(20)
x = np.exp(-0.15 * n) * np.sin(2 * np.pi * 0.12 * n)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))

# Homogeneity failure: scale the input by 2, see how the output scales.
for a_, style in [(1.0, "-o"), (2.0, "-s")]:
    ax[0].plot(n, sys_square(a_ * x), style, ms=4, label=f"T{{{a_:g}x}}")
ax[0].plot(n, 2.0 * sys_square(x), "--^", ms=4, label="2·T{x}")
ax[0].set_title("x[n]² : T{2x} ≠ 2·T{x}  (it is 4·T{x})")
ax[0].legend(fontsize=8); ax[0].set_xlabel("n")

# Additivity failure for the bias system.
x1, x2 = np.sin(0.4 * n), 0.5 * np.cos(0.25 * n)
ax[1].plot(n, sys_bias(x1 + x2), "-o", ms=4, label="T{x₁+x₂}")
ax[1].plot(n, sys_bias(x1) + sys_bias(x2), "--s", ms=4, label="T{x₁}+T{x₂}")
ax[1].set_title("x[n]+2 : the offset is counted twice on the right")
ax[1].legend(fontsize=8); ax[1].set_xlabel("n")
fig.tight_layout()

---
## 3. Time invariance, tested numerically

A system is **time-invariant** if delaying the input delays the output by the same amount and
changes nothing else:

$$y[n] = T\{x[n]\} \quad\Longrightarrow\quad y[n-d] = T\{x[n-d]\}$$

To test this we need a delay that does not wrap around. Zero-pad at the front, and compare on
the region where both signals are defined.

In [ ]:
def delay(x, d):
    """Delay by d samples with zero padding (NOT np.roll, which wraps)."""
    return np.concatenate([np.zeros(d), x])

def is_time_invariant(system, n=64, d=7, trials=20, tol=1e-9, rng=None):
    """Check that delaying the input delays the output and changes nothing else.

    We feed the delayed input, then strip the first d samples to realign, and compare
    against the undelayed output. Note what we do NOT do: compare against a zero-padded
    copy of the output. Padding with zeros presumes T{0} = 0, which holds for linear
    systems but not for, say, y[n] = x[n] + 2 -- and that system really is time-invariant.
    """
    rng = np.random.default_rng(1) if rng is None else rng
    worst = 0.0
    for _ in range(trials):
        x = rng.normal(size=n)
        shifted = system(delay(x, d))[d:]  # T{x[n-d]}, realigned to start at n = 0
        direct = system(x)                 # y[n]
        m = min(len(shifted), len(direct))
        worst = max(worst, np.max(np.abs(shifted[:m] - direct[:m])))
    return worst < tol, worst

print(f"{'system':<20} {'linear?':<9} {'time-inv?':<11} {'LTI?':<6}")
print("-" * 50)
for name, f in SYSTEMS.items():
    lin, _ = is_linear(f)
    ti, _ = is_time_invariant(f)
    print(f"{name:<20} {str(lin):<9} {str(ti):<11} {str(lin and ti):<6}")

> **`n*x[n]` is the instructive failure.** It is perfectly linear, so the §2 test cleared it.
> But its behaviour depends on *when* the input arrives — the same pulse gets a bigger gain if
> it happens later. Linearity alone does not buy you convolution; you need both properties.
>
> **`x[n] + 2` passes this test**, exactly as the theory says it should: shifting the input
> shifts the output and changes nothing else. It fails to be LTI on the *linearity* count
> alone. Systems like this are called **affine**, and the standard move is to analyse the
> deviation from the operating point — subtract the 2, analyse what is left as an LTI system,
> add the 2 back at the end.
>
> Getting this right required care in the test itself. See the docstring above: comparing
> `T{x[n−d]}` against a *zero-padded* copy of `y[n]` would have wrongly flagged `x[n] + 2` as
> time-varying, because zero-padding quietly assumes `T{0} = 0`. That assumption is safe for
> linear systems and wrong here.

In [ ]:
d = 8
pulse = np.zeros(40); pulse[5:10] = 1.0

fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))
for a_, sys_f, name in [(ax[0], sys_movavg, "moving average (LTI)"),
                        (ax[1], sys_ramp_gain, "n·x[n] (linear, time-VARYING)"),
                        (ax[2], sys_bias, "x[n]+2 (time-invariant, not linear)")]:
    shifted = sys_f(delay(pulse, d))[d:]      # realigned
    direct = sys_f(pulse)
    m = min(len(shifted), len(direct))
    a_.plot(np.arange(m), shifted[:m], "-o", ms=4, label="T{x[n−d]}, realigned")
    a_.plot(np.arange(m), direct[:m], "--s", ms=4, label="y[n]")
    a_.set_title(name, fontsize=10); a_.legend(fontsize=8); a_.set_xlabel("n")
fig.tight_layout()

---
## 4. The impulse as a basis

Here is the idea that makes everything work. Any discrete signal can be written as a weighted
sum of shifted unit impulses:

$$x[n] = \sum_{k=-\infty}^{\infty} x[k]\,\delta[n-k]$$

This looks like a tautology — and it is. It says "the signal equals the sum of its samples,
each placed at its own position." That triviality is exactly what makes it useful: it is a
decomposition into a basis, and it costs nothing to write down.

In [ ]:
x = np.array([2.0, -1.0, 3.0, 1.5])
n = np.arange(len(x))

fig, ax = plt.subplots(1, 5, figsize=(14, 2.4), sharey=True)
recon = np.zeros_like(x)
for k in range(len(x)):
    comp = np.zeros_like(x)
    comp[k] = x[k]
    recon += comp
    stem_ax(ax[k], n, comp, color="tab:orange")
    ax[k].set_title(f"x[{k}]·δ[n−{k}] = {x[k]:g}·δ[n−{k}]", fontsize=9)
    ax[k].set_xlabel("n")
stem_ax(ax[4], n, recon)
ax[4].set_title("sum of all four = x[n]", fontsize=9)
ax[4].set_xlabel("n")
fig.tight_layout()

print("reconstruction exact:", np.array_equal(recon, x))

---
## 5. Deriving convolution

Now apply the system to that decomposition. Each step uses exactly one property, and it is
worth naming them as you go:

$$
\begin{aligned}
y[n] &= T\Big\{\sum_k x[k]\,\delta[n-k]\Big\} \\[4pt]
     &= \sum_k T\big\{x[k]\,\delta[n-k]\big\} && \text{additivity} \\[4pt]
     &= \sum_k x[k]\; T\big\{\delta[n-k]\big\} && \text{homogeneity (}x[k]\text{ is a constant)} \\[4pt]
     &= \sum_k x[k]\; h[n-k] && \text{time invariance, with } h = T\{\delta\}
\end{aligned}
$$

That last line is the **convolution sum**, written $y = x * h$.

Read the derivation again and notice how little it took. Additivity let us push $T$ inside the
sum. Homogeneity let us pull the constant $x[k]$ out. Time invariance let us write
$T\{\delta[n-k]\}$ as $h[n-k]$ — the *same* function $h$, merely shifted, rather than a
different response for every $k$. Remove any one property and the chain breaks.

**The impulse response $h[n] = T\{\delta[n]\}$ is therefore a complete description of an LTI
system.** Nothing else is needed.

In [ ]:
def convolve_by_hand(x, h):
    """Direct transcription of y[n] = sum_k x[k] h[n-k]."""
    Ny = len(x) + len(h) - 1
    y = np.zeros(Ny)
    for n_ in range(Ny):
        total = 0.0
        for k in range(len(x)):
            if 0 <= n_ - k < len(h):        # h is zero outside its support
                total += x[k] * h[n_ - k]
        y[n_] = total
    return y

rng = np.random.default_rng(2)
x_test, h_test = rng.normal(size=25), rng.normal(size=8)

mine = convolve_by_hand(x_test, h_test)
theirs = np.convolve(x_test, h_test)

print("max difference from np.convolve:", np.max(np.abs(mine - theirs)))
print("identical:", np.allclose(mine, theirs))

---
## 6. The black-box experiment

The claim deserves a real test. Below is a system whose internals you are not allowed to look
at. We will:

1. Feed it a single impulse and record $h$.
2. Use *only* that $h$ to predict its response to a completely different signal.
3. Compare against the truth.

If LTI theory works, step 2 should match step 3 to numerical precision.

In [ ]:
def black_box(x):
    """Some LTI system. Pretend you cannot read this line."""
    return signal.lfilter([0.5, 0.3, -0.1], [1.0, -0.4, 0.08], x)

N = 200

# Step 1: measure the impulse response.
delta = signal.unit_impulse(N)
h_measured = black_box(delta)

# Step 2: predict the response to an arbitrary input using only h_measured.
x_new = (signal.square(2 * np.pi * 0.03 * np.arange(N))
         + 0.4 * np.sin(2 * np.pi * 0.11 * np.arange(N)))
y_predicted = np.convolve(x_new, h_measured)[:N]

# Step 3: the truth.
y_actual = black_box(x_new)

err = np.max(np.abs(y_predicted - y_actual))
print(f"maximum prediction error over {N} samples: {err:.3e}")
print(f"signal peak amplitude for scale           : {np.max(np.abs(y_actual)):.3f}")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
stem_ax(ax[0], np.arange(30), h_measured[:30])
ax[0].set_title("Step 1: measured impulse response h[n] (first 30 samples)")

ax[1].plot(x_new, "0.6", lw=1, label="new input x[n]")
ax[1].plot(y_actual, lw=1.6, label="actual output")
ax[1].plot(y_predicted, "--", lw=1.2, label="predicted from h alone")
ax[1].legend(fontsize=8, ncol=3)
ax[1].set_title("Step 2 vs 3: prediction lies exactly on the truth")

ax[2].semilogy(np.abs(y_predicted - y_actual) + 1e-18, lw=0.8, color="tab:red")
ax[2].set_title("absolute error (log scale) -- this is floating-point noise")
ax[2].set_xlabel("n")
fig.tight_layout()

> **The error is around $10^{-16}$** — that is double-precision round-off, not a modelling
> error. One impulse measurement predicted the response to a signal it had never seen.
>
> **Why it is not *exactly* zero:** `h_measured` is truncated at 200 samples, and this system
> is IIR so its true impulse response never fully ends. The tail we discarded is around
> $10^{-30}$ by then, so it does not matter here. For a slowly-decaying system it would.

### The same experiment on a non-LTI system

Now repeat it with `x[n]^2`. The procedure runs without complaint — that is the danger.

In [ ]:
h_bad = sys_square(delta)                       # "impulse response" of a non-LTI system
y_pred_bad = np.convolve(x_new, h_bad)[:N]
y_true_bad = sys_square(x_new)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(y_true_bad, lw=1.5, label="actual output of x[n]²")
ax.plot(y_pred_bad, "--", lw=1.5, label="convolution prediction")
ax.legend(); ax.set_xlabel("n")
ax.set_title("Non-LTI: the prediction is simply wrong")

print(f"max error: {np.max(np.abs(y_pred_bad - y_true_bad)):.3f}"
      f"   (compare with {err:.1e} for the LTI system)")
print("\nNote that h_bad = delta^2 = delta, so convolution predicted y = x. It does not.")

---
## 7. Flip-and-slide, step by step

Exams ask you to convolve two short sequences by hand. The mechanical recipe follows directly
from $y[n] = \sum_k x[k]\,h[n-k]$:

1. Plot $x[k]$ against $k$.
2. Plot $h[-k]$ — the impulse response **time-reversed**.
3. Shift the reversed copy right by $n$ to get $h[n-k]$.
4. Multiply point-by-point and sum. That single number is $y[n]$.
5. Increment $n$ and repeat.

The "flip" in step 2 is not a convention someone chose; it comes from the $n-k$ in the index.
Here is the whole process laid out.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 2.0])
h = np.array([1.0, 0.5, 0.25])
y = np.convolve(x, h)

def sample_at(arr, idx):
    """arr[idx] with zeros outside the array's support."""
    out = np.zeros(len(idx), dtype=float)
    inside = (idx >= 0) & (idx < len(arr))
    out[inside] = arr[idx[inside]]
    return out

k = np.arange(-3, 7)

fig, ax = plt.subplots(2, 3, figsize=(14, 5.5), sharex=True, sharey=True)
for n_, a_ in enumerate(ax.ravel()):
    xk = sample_at(x, k)
    hk = sample_at(h, n_ - k)          # h[n-k]: flipped and shifted by n
    prod = xk * hk

    a_.stem(k, xk, basefmt=" ", linefmt="tab:blue", markerfmt="o", label="x[k]")
    a_.stem(k, hk, basefmt=" ", linefmt="tab:orange", markerfmt="s", label=f"h[{n_}−k]")
    a_.bar(k, prod, width=0.35, color="tab:green", alpha=0.7, label="product")
    a_.axhline(0, color="k", lw=0.7)
    a_.set_title(f"n = {n_}:  y[{n_}] = {prod.sum():.3f}", fontsize=10)
    a_.set_xticks(k[::2])
    if n_ == 0:
        a_.legend(fontsize=8, loc="upper left")
for a_ in ax[-1]:
    a_.set_xlabel("k")
fig.suptitle("Flip-and-slide: the green bars are what you sum at each n", y=1.01)
fig.tight_layout()

print("y from the panels:", [round(float(np.sum(sample_at(x, k) * sample_at(h, i - k))), 4)
                             for i in range(6)])
print("np.convolve      :", y)

### Two ways to organise the same arithmetic

The **overlap table** is how most textbooks show it: write out $x[k]h[n-k]$ for every pair and
sum along diagonals. This is also, exactly, polynomial multiplication.

In [ ]:
outer = np.outer(x, h)          # outer[i, j] = x[i] * h[j], contributing to y[i+j]
print("outer product x[i]·h[j]:")
print(outer)
print()
print("y[n] = sum of the anti-diagonal where i + j = n:")
for n_ in range(len(x) + len(h) - 1):
    terms = [(i, n_ - i) for i in range(len(x)) if 0 <= n_ - i < len(h)]
    expr = " + ".join(f"{x[i]:g}·{h[j]:g}" for i, j in terms)
    print(f"  y[{n_}] = {expr:<28} = {sum(x[i] * h[j] for i, j in terms):.4f}")

print("\nsame as polynomial multiplication:")
print("  np.convolve      :", np.convolve(x, h))
print("  polynomial polymul:", np.polynomial.polynomial.polymul(x, h))

> **Convolution is polynomial multiplication.** If $X(z) = 1 + 2z + 3z^2 + 2z^3$ and
> $H(z) = 1 + 0.5z + 0.25z^2$, then the coefficients of $X(z)H(z)$ are $x * h$. This is not an
> analogy — it is the same computation, and it is *why* the z-transform turns convolution into
> multiplication.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

x_a = np.zeros(50); x_a[8:24] = 1.0
h_a = np.exp(-np.arange(18) / 5.0)
y_a = np.convolve(x_a, h_a)

fig, ax = plt.subplots(2, 1, figsize=(9, 4.6))
ax[0].plot(np.arange(len(x_a)), x_a, "tab:blue", lw=1.5, label="x[k]")
flip, = ax[0].plot([], [], "tab:orange", lw=1.5, label="h[n−k]")
fill = ax[0].fill_between([], [], color="tab:green", alpha=0.4)
ax[0].set_xlim(-20, 70); ax[0].set_ylim(-0.1, 1.4)
ax[0].legend(fontsize=8, loc="upper right"); ax[0].set_xlabel("k")

out, = ax[1].plot([], [], "tab:green", lw=1.8)
marker, = ax[1].plot([], [], "ro", ms=6)
ax[1].set_xlim(-20, 70); ax[1].set_ylim(0, y_a.max() * 1.15)
ax[1].set_xlabel("n"); ax[1].set_ylabel("y[n]")

def frame(n_):
    kk = np.arange(-20, 70)
    hk = sample_at(h_a, n_ - kk)
    flip.set_data(kk, hk)
    out.set_data(np.arange(n_ + 1), y_a[:n_ + 1])
    marker.set_data([n_], [y_a[n_]])
    ax[0].set_title(f"n = {n_},  overlap sum = {y_a[n_]:.3f}", fontsize=10)
    return flip, out, marker

anim = animation.FuncAnimation(fig, frame, frames=len(y_a), interval=80, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

---
## 8. Convolution modes and support

If $x$ has $N$ samples and $h$ has $M$, then $x*h$ has $N+M-1$ samples. The output is longer
than the input, and this is not an artefact — a filter genuinely rings on after the input stops.

`np.convolve` and `scipy.signal.convolve` offer three ways to deal with that.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
h = np.ones(3) / 3

for mode in ["full", "same", "valid"]:
    out = np.convolve(x, h, mode=mode)
    print(f"{mode:<6} len={len(out)}  {out}")

print()
print(f"full  : N + M - 1 = {len(x)} + {len(h)} - 1 = {len(x) + len(h) - 1}  -- the complete result")
print(f"same  : N         = {len(x)}                  -- centred slice, for keeping alignment")
print(f"valid : N - M + 1 = {len(x)} - {len(h)} + 1 = {len(x) - len(h) + 1}  -- only fully-overlapped points")

| mode | length | keep it when |
|------|--------|--------------|
| `full` | $N+M-1$ | you want the mathematically complete answer, including the tail |
| `same` | $N$ | you are filtering and want the output to line up with the input |
| `valid` | $N-M+1$ | edge effects are unacceptable and you can afford to lose samples |

**The edges are where the arguments happen.** `full` and `same` implicitly assume the signal
is zero outside its support, which produces a fade-in and fade-out that is not in your data.
`valid` refuses to guess, at the cost of $M-1$ samples.

In [ ]:
n = np.arange(120)
sig = 2.0 + np.sin(2 * np.pi * 0.03 * n)      # note the DC offset of 2
h_ma = np.ones(21) / 21

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(n, sig, "0.7", lw=1.2, label="input (mean = 2)")
ax.plot(np.convolve(sig, h_ma, mode="same"), lw=1.5, label="mode='same' -- dips at both ends")
valid = np.convolve(sig, h_ma, mode="valid")
ax.plot(np.arange(len(h_ma) // 2, len(h_ma) // 2 + len(valid)), valid,
        lw=1.5, label="mode='valid' -- shorter but honest")
ax.legend(fontsize=8); ax.set_xlabel("n")
ax.set_title("Zero-padding at the edges pulls the average toward zero")

> **That dip is a real bug source.** A 21-tap moving average on a signal with mean 2 reports
> values near 0 at the first and last few samples, purely because the padding is zeros. If you
> then compute statistics on the filtered signal, the edges corrupt them. Options: use
> `valid`, or pad with edge values (`np.pad(..., mode="edge")`) or by reflection, which is
> what `scipy.signal.filtfilt` does by default.

In [ ]:
padded = np.pad(sig, len(h_ma) // 2, mode="edge")
edge_filtered = np.convolve(padded, h_ma, mode="valid")

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(n, sig, "0.7", lw=1.2, label="input")
ax.plot(np.convolve(sig, h_ma, mode="same"), lw=1.2, label="zero padding")
ax.plot(edge_filtered, lw=1.8, label="edge padding -- no artefact")
ax.legend(fontsize=8); ax.set_xlabel("n"); ax.set_xlim(0, 40)
ax.set_title("Padding choice, zoomed on the leading edge")

---
## 9. Properties of convolution

These are the identities you will use to simplify block diagrams. Each one is stated, then
checked numerically — and checking them yourself is a genuinely good way to remember them.

In [ ]:
rng = np.random.default_rng(3)
x1, x2 = rng.normal(size=40), rng.normal(size=40)
h1, h2 = rng.normal(size=7), rng.normal(size=5)
a, b = 2.5, -1.3

def pad_to_match(u, v):
    """Zero-pad the shorter array so the two can be added or compared."""
    L = max(len(u), len(v))
    return np.pad(u, (0, L - len(u))), np.pad(v, (0, L - len(v)))

def check(name, lhs, rhs):
    lhs, rhs = pad_to_match(lhs, rhs)
    err = np.max(np.abs(lhs - rhs))
    print(f"{name:<48} {str(np.allclose(lhs, rhs)):<6} (max err {err:.2e})")

check("commutative   x*h = h*x",
      np.convolve(x1, h1),
      np.convolve(h1, x1))

check("associative   (x*h1)*h2 = x*(h1*h2)",
      np.convolve(np.convolve(x1, h1), h2),
      np.convolve(x1, np.convolve(h1, h2)))

h1p, h2p = pad_to_match(h1, h2)
r1, r2 = pad_to_match(np.convolve(x1, h1), np.convolve(x1, h2))
check("distributive  x*(h1+h2) = x*h1 + x*h2",
      np.convolve(x1, h1p + h2p),
      r1 + r2)

check("identity      x*delta = x",
      np.convolve(x1, np.array([1.0])),
      x1)

check("linearity     (a·x1 + b·x2)*h = a(x1*h) + b(x2*h)",
      np.convolve(a * x1 + b * x2, h1),
      a * np.convolve(x1, h1) + b * np.convolve(x2, h1))

In [ ]:
# The shift property: convolving with a shifted impulse delays the signal.
x_s = np.array([1.0, 2.0, 3.0])
for d in [0, 1, 4]:
    shifted_delta = signal.unit_impulse(d + 1, d)
    print(f"x * delta[n-{d}] = {np.convolve(x_s, shifted_delta)}")
print("\nSo a pure delay is an LTI system with h[n] = delta[n-d].")

In [ ]:
# Convolving with the unit step performs a running sum (discrete integration).
x_r = np.array([1.0, -2.0, 3.0, 0.5, -1.0])
u = np.ones(len(x_r))

print("x            :", x_r)
print("x * u (first N):", np.convolve(x_r, u)[:len(x_r)])
print("np.cumsum(x) :", np.cumsum(x_r))
print("\nidentical:", np.allclose(np.convolve(x_r, u)[:len(x_r)], np.cumsum(x_r)))
print("\nThe accumulator is the LTI system with h[n] = u[n].")

---
## 10. Cascade and parallel systems

Associativity and distributivity have direct engineering meaning, and this is where the algebra
starts paying rent.

- **Cascade** (series): $h_{\text{total}} = h_1 * h_2$. Because convolution commutes, **the
  order of LTI blocks does not matter.**
- **Parallel** (sum of branches): $h_{\text{total}} = h_1 + h_2$.

In [ ]:
h_lp = signal.firwin(31, 0.2)                      # a low-pass
h_diff = np.array([1.0, -1.0])                     # a differencer

h_cascade = np.convolve(h_lp, h_diff)
h_parallel = h_lp + np.pad(h_diff, (0, len(h_lp) - len(h_diff)))

x = np.cumsum(np.random.default_rng(4).normal(size=300))   # a random walk

# Verify the cascade identity two ways.
route_a = np.convolve(np.convolve(x, h_lp), h_diff)
route_b = np.convolve(np.convolve(x, h_diff), h_lp)        # swapped order
route_c = np.convolve(x, h_cascade)                        # single combined filter

print("filter order swapped, same result :", np.allclose(route_a, route_b))
print("combined h gives same result      :", np.allclose(route_a, route_c))
print(f"h_cascade length = {len(h_lp)} + {len(h_diff)} - 1 = {len(h_cascade)}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))

w, H_lp = signal.freqz(h_lp, worN=2048)
_, H_diff = signal.freqz(h_diff, worN=2048)
_, H_casc = signal.freqz(h_cascade, worN=2048)
_, H_par = signal.freqz(h_parallel, worN=2048)

for H, name in [(H_lp, "h₁ low-pass"), (H_diff, "h₂ differencer"),
                (H_casc, "cascade h₁*h₂"), (H_par, "parallel h₁+h₂")]:
    ax[0].plot(w / np.pi, 20 * np.log10(np.abs(H) + 1e-12), label=name)
ax[0].set(xlabel="normalised frequency (1 = Nyquist)", ylabel="|H| [dB]", ylim=(-80, 15))
ax[0].legend(fontsize=8); ax[0].set_title("Cascade multiplies magnitudes (adds dB)")

# Confirm the dB addition numerically at one frequency.
idx = 400
print(f"at ω/π = {w[idx]/np.pi:.3f}")
print(f"  |H1| dB + |H2| dB = {20*np.log10(abs(H_lp[idx])):.3f} + {20*np.log10(abs(H_diff[idx])):.3f}"
      f" = {20*np.log10(abs(H_lp[idx])) + 20*np.log10(abs(H_diff[idx])):.3f}")
print(f"  |H_cascade| dB    = {20*np.log10(abs(H_casc[idx])):.3f}")

stem_ax(ax[1], np.arange(len(h_cascade)), h_cascade)
ax[1].set_title("h_cascade[n] = h₁ * h₂"); ax[1].set_xlabel("n")
fig.tight_layout()

> **"Order does not matter" is true mathematically and false practically.** With ideal
> arithmetic, a low-pass followed by a differencer is identical to the reverse. With finite
> precision or fixed-point hardware, putting the high-gain stage first can overflow or amplify
> quantisation noise that the later stage cannot remove. This is exactly why `scipy` offers
> second-order sections and why filter *implementation* is a separate topic from filter
> *design*.

---
## 11. Continuous-time convolution

In continuous time the sum becomes an integral:

$$y(t) = \int_{-\infty}^{\infty} x(\tau)\,h(t-\tau)\,d\tau$$

Numerically you approximate it with a discrete convolution scaled by the sample spacing:

$$y(t) \approx \Delta t \sum_k x[k]\,h[n-k]$$

**Forgetting the $\Delta t$ is the single most common error here.** Without it your answer is
wrong by a factor of the sampling rate, and it will look plausible because the shape is right.

In [ ]:
dt = 0.001
t = np.arange(0, 4, dt)

def rect(t, start, stop):
    return ((t >= start) & (t < stop)).astype(float)

x_c = rect(t, 0, 1)          # unit rectangle, width 1
h_c = rect(t, 0, 1)          # same

y_c = np.convolve(x_c, h_c) * dt          # <-- the dt matters
t_y = np.arange(len(y_c)) * dt

# Analytic result: rect(width 1) * rect(width 1) = triangle, base [0,2], peak 1 at t=1.
y_exact = np.clip(1 - np.abs(t_y - 1), 0, None)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
ax[0].plot(t, x_c, label="x(t) = rect"); ax[0].plot(t, h_c, "--", label="h(t) = rect")
ax[0].set(xlim=(-0.2, 2.5), xlabel="t [s]"); ax[0].legend(); ax[0].set_title("inputs")
ax[1].plot(t_y, y_c, lw=2.5, label="numerical, ×dt")
ax[1].plot(t_y, y_exact, "r--", lw=1.2, label="analytic triangle")
ax[1].plot(t_y, np.convolve(x_c, h_c)[:len(t_y)], ":", color="0.5",
           label="forgot the dt (1000× too big)")
ax[1].set(xlim=(-0.2, 2.5), xlabel="t [s]"); ax[1].legend(fontsize=8)
ax[1].set_title("rect * rect = triangle")

print(f"peak of numerical result : {y_c.max():.6f}   (analytic: 1.0)")
print(f"max error vs analytic    : {np.max(np.abs(y_c - y_exact)):.2e}")

In [ ]:
# A second worked example with a known closed form:
#   x(t) = u(t),  h(t) = e^{-at} u(t)   ->   y(t) = (1 - e^{-at}) / a
a_rate = 3.0
x_step = (t >= 0).astype(float)
h_exp = np.exp(-a_rate * t) * (t >= 0)

y_num = np.convolve(x_step, h_exp)[:len(t)] * dt
y_ana = (1 - np.exp(-a_rate * t)) / a_rate

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(t, h_exp); ax[0].set(xlim=(0, 2), xlabel="t [s]", title="h(t) = e^{−3t}u(t)")
ax[1].plot(t, y_num, lw=2.5, label="numerical")
ax[1].plot(t, y_ana, "r--", lw=1.2, label="(1 − e^{−3t})/3")
ax[1].axhline(1 / a_rate, color="k", ls=":", lw=1, label="final value 1/a")
ax[1].set(xlim=(0, 2), xlabel="t [s]", title="step response"); ax[1].legend(fontsize=8)
fig.tight_layout()

print(f"max error: {np.max(np.abs(y_num - y_ana)):.2e}  (limited by the dt discretisation)")

> **Convergence check.** The error above is set by `dt`, not by anything conceptual. Halving
> `dt` should roughly halve the error for this first-order scheme. Verifying that is a useful
> habit whenever you approximate an integral — if refining the grid does not improve the
> answer, your discretisation is not doing what you think.

In [ ]:
print("dt        max error")
for dt_ in [0.02, 0.01, 0.005, 0.0025]:
    tt = np.arange(0, 4, dt_)
    yn = np.convolve((tt >= 0).astype(float), np.exp(-a_rate * tt) * (tt >= 0))[:len(tt)] * dt_
    ya = (1 - np.exp(-a_rate * tt)) / a_rate
    print(f"{dt_:<9.4f} {np.max(np.abs(yn - ya)):.3e}")
print("\nError falls roughly linearly with dt, as expected for this scheme.")

---
## 12. Causality, memory, and stability

Three properties, each readable directly off the impulse response.

| Property | Condition on $h$ | Meaning |
|----------|------------------|---------|
| **Causal** | $h[n] = 0$ for $n < 0$ | output never depends on future input |
| **Memoryless** | $h[n] = c\,\delta[n]$ | output depends only on the present sample |
| **BIBO stable** | $\sum_n \lvert h[n]\rvert < \infty$ | bounded input always gives bounded output |
| **FIR** | $h$ has finite support | always stable |
| **IIR** | $h$ has infinite support | stable only if it decays fast enough |

In [ ]:
# Define each impulse response as a function of n so we can evaluate it over any range.
IMPULSE_RESPONSES = {
    "0.9ⁿ u[n]  (stable IIR)":   lambda k: 0.9 ** k,
    "1.05ⁿ u[n] (unstable IIR)": lambda k: 1.05 ** k,
    "1/(n+1)    (marginal)":     lambda k: 1.0 / (k + 1.0),
    "1/(n+1)²   (stable)":       lambda k: 1.0 / (k + 1.0) ** 2,
    "FIR, 8 taps":               lambda k: (k < 8).astype(float),
}

# The honest test is not "what is the sum" but "does the sum settle down as N grows".
lengths = [10, 100, 1_000, 10_000, 100_000]
header = "".join(f"{N:>13}" for N in lengths)
print(f"{'h[n]':<28}{header}")
print("-" * (28 + 13 * len(lengths)))
for name, hfunc in IMPULSE_RESPONSES.items():
    with np.errstate(over="ignore"):
        sums = [np.sum(np.abs(hfunc(np.arange(N)))) for N in lengths]
    print(f"{name:<28}" + "".join(f"{s:>13.4g}" for s in sums))

n = np.arange(0, 60)
systems_h = {name: hfunc(n) for name, hfunc in IMPULSE_RESPONSES.items()}

> **`1/(n+1)` is the trap.** Its impulse response decays to zero, so it *looks* stable, but
> $\sum 1/n$ is the harmonic series, which diverges. Read along its row: 2.9, 5.2, 7.5, 9.8,
> 12.1 — each ten-fold increase in $N$ adds a constant, which is exactly $\ln N$ growth. Slow
> enough that a short simulation would never reveal it. "The impulse response goes to zero" is
> **not** the stability condition; absolute summability is.
>
> **`1/(n+1)²` settles at 1.6449**, and that number is $\pi^2/6$ — the Basel sum. Getting a
> recognisable closed form out of the numerics is a free confirmation that the computation is
> doing what the mathematics says it should.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.2))

for name, h_ in systems_h.items():
    ax[0].semilogy(n, np.abs(h_) + 1e-20, lw=1.2, label=name)
ax[0].set(xlabel="n", ylabel="|h[n]|", title="impulse responses (log scale)")
ax[0].legend(fontsize=7)

for name, h_ in systems_h.items():
    ax[1].plot(n, np.cumsum(np.abs(h_)), lw=1.2, label=name)
ax[1].set(xlabel="n", ylabel="cumulative Σ|h|", title="running absolute sum", ylim=(0, 30))

# Demonstrate instability: bounded input, unbounded output.
x_bounded = np.sign(np.sin(2 * np.pi * 0.05 * np.arange(80)))     # always ±1
y_stable = np.convolve(x_bounded, 0.9 ** np.arange(60))[:80]
y_unstable = np.convolve(x_bounded, 1.05 ** np.arange(60))[:80]
ax[2].plot(y_stable, label="stable h (bounded out)")
ax[2].plot(y_unstable, label="unstable h (grows)")
ax[2].set(xlabel="n", title="bounded input ±1, two systems")
ax[2].legend(fontsize=8)
fig.tight_layout()

print(f"input is bounded by      : {np.max(np.abs(x_bounded)):.1f}")
print(f"stable system output max : {np.max(np.abs(y_stable)):.2f}")
print(f"unstable system output max: {np.max(np.abs(y_unstable)):.2f}")

### Non-causal filters are useful — offline

A non-causal $h$ cannot be built in real time, but there is nothing wrong with it when you
already have the whole recording on disk. A centred moving average and `filtfilt` are both
non-causal, and both are standard practice in offline analysis.

In [ ]:
n_c = np.arange(-10, 11)
h_causal = np.where((n_c >= 0) & (n_c < 5), 0.2, 0.0)
h_noncausal = np.where(np.abs(n_c) <= 2, 0.2, 0.0)      # centred: uses "future" samples

fig, ax = plt.subplots(1, 2, figsize=(12, 2.8), sharey=True)
stem_ax(ax[0], n_c, h_causal); ax[0].set_title("causal: h[n] = 0 for n < 0")
stem_ax(ax[1], n_c, h_noncausal, color="tab:red")
ax[1].set_title("non-causal: needs samples that have not arrived yet")
for a_ in ax:
    a_.axvline(0, color="k", ls=":", lw=1); a_.set_xlabel("n")
fig.tight_layout()

# The practical consequence: delay.
x_edge = np.concatenate([np.zeros(20), np.ones(30)])
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.plot(x_edge, "0.6", lw=1.5, label="input step")
ax.plot(np.convolve(x_edge, h_causal, mode="same"), lw=1.5, label="causal -> delayed")
ax.plot(np.convolve(x_edge, h_noncausal, mode="same"), lw=1.5, label="centred -> aligned")
ax.legend(fontsize=8); ax.set_xlim(10, 35); ax.set_xlabel("n")
ax.set_title("Causality costs you delay")

---
## 13. Step response ↔ impulse response

The step response $s[n] = T\{u[n]\}$ is often far easier to measure than the impulse response —
you can flip a switch, but generating a true impulse is hard. Fortunately the two are related
by a difference (discrete) or a derivative (continuous):

$$h[n] = s[n] - s[n-1], \qquad h(t) = \frac{d}{dt}s(t)$$

and conversely $s[n] = \sum_{k \le n} h[k]$.

In [ ]:
h_true = black_box(signal.unit_impulse(120))
s_meas = black_box(np.ones(120))                  # measure the step response instead

h_from_step = np.diff(s_meas, prepend=0.0)        # h[n] = s[n] - s[n-1]
s_from_h = np.cumsum(h_true)                      # s[n] = sum of h up to n

print("h recovered from step response :", np.allclose(h_from_step, h_true))
print("s recovered from impulse resp. :", np.allclose(s_from_h, s_meas))
print(f"max errors: {np.max(np.abs(h_from_step - h_true)):.2e}, "
      f"{np.max(np.abs(s_from_h - s_meas)):.2e}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3))
stem_ax(ax[0], np.arange(25), s_meas[:25], color="tab:green")
ax[0].set_title("measured step response s[n]"); ax[0].set_xlabel("n")
stem_ax(ax[1], np.arange(25), h_from_step[:25])
ax[1].set_title("h[n] = s[n] − s[n−1], recovered exactly"); ax[1].set_xlabel("n")
fig.tight_layout()

> **In a real lab this is how you do it.** Differentiating a measured step response amplifies
> noise (differencing is a high-pass operation), so you smooth first or fit a model. But the
> principle — measure the easy thing, derive the useful thing — is why step response
> specifications dominate datasheets.

---
## 14. Complex exponentials are eigenfunctions

Here is the property that motivates the entire frequency-domain half of the course. Feed
$x[n] = e^{j\omega n}$ into an LTI system:

$$y[n] = \sum_k h[k]\,e^{j\omega(n-k)} = e^{j\omega n}\underbrace{\sum_k h[k]e^{-j\omega k}}_{H(e^{j\omega})}$$

The output is **the same signal, scaled by a complex number**. In linear-algebra language,
complex exponentials are eigenfunctions of every LTI system and $H(e^{j\omega})$ is the
eigenvalue.

This is why a sinusoid in gives a sinusoid out at the same frequency, with only amplitude and
phase changed. No LTI system can create a new frequency.

In [ ]:
h = np.array([0.3, 0.25, 0.2, 0.15, 0.1])
n = np.arange(80)

for omega in [0.2, 0.7, 1.5]:
    x_exp = np.exp(1j * omega * n)
    y_exp = np.convolve(x_exp, h)[:len(n)]

    # Skip the first len(h)-1 samples: the transient before full overlap.
    ratio = y_exp[len(h):] / x_exp[len(h):]
    H_theory = np.sum(h * np.exp(-1j * omega * np.arange(len(h))))

    print(f"ω = {omega}")
    print(f"  y/x is constant     : {np.allclose(ratio, ratio[0])}   (std = {np.std(ratio):.2e})")
    print(f"  measured eigenvalue : {ratio[0]:.6f}")
    print(f"  H(e^jω) from formula: {H_theory:.6f}")
    print(f"  |H| = {abs(H_theory):.4f}, phase = {np.degrees(np.angle(H_theory)):.2f}°")
    print()

In [ ]:
# The real-signal version: amplitude scaled, phase shifted, frequency unchanged.
omega = 0.7
x_cos = np.cos(omega * n)
y_cos = np.convolve(x_cos, h)[:len(n)]
H = np.sum(h * np.exp(-1j * omega * np.arange(len(h))))
y_predicted = np.abs(H) * np.cos(omega * n + np.angle(H))

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(n, x_cos, "0.65", lw=1.3, label="input cos(0.7n)")
ax.plot(n, y_cos, lw=1.8, label="output (convolution)")
ax.plot(n, y_predicted, "r--", lw=1.2, label="|H|·cos(0.7n + ∠H)")
ax.set_xlim(10, 80); ax.legend(fontsize=8, ncol=3); ax.set_xlabel("n")
ax.set_title("Sinusoid in, same-frequency sinusoid out")

print(f"max error after the transient: {np.max(np.abs(y_cos[len(h):] - y_predicted[len(h):])):.2e}")

In [ ]:
# And the spectral proof that no new frequencies appear.
x_two = np.cos(0.3 * np.arange(512)) + 0.5 * np.cos(1.1 * np.arange(512))
y_two = np.convolve(x_two, h)[:512]

f = np.fft.rfftfreq(512)
fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogy(f, np.abs(np.fft.rfft(x_two)) + 1e-12, lw=1, label="input spectrum")
ax.semilogy(f, np.abs(np.fft.rfft(y_two)) + 1e-12, lw=1, label="output spectrum")
ax.set(xlabel="normalised frequency", ylabel="magnitude", ylim=(1e-2, 1e3))
ax.legend(); ax.set_title("Two lines in, the same two lines out -- rescaled, never moved")

> **Contrast with the squarer.** `x[n]²` applied to `cos(ω n)` produces DC and $2\omega$
> components that were not in the input. Generating new frequencies is the signature of a
> non-linear system — and it is how distortion, mixers, and harmonic generators work.

In [ ]:
y_sq = sys_square(x_two)
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.semilogy(f, np.abs(np.fft.rfft(x_two)) + 1e-12, lw=1, label="input")
ax.semilogy(f, np.abs(np.fft.rfft(y_sq)) + 1e-12, lw=1, label="after squaring")
# Squaring two tones at w1=0.3 and w2=1.1 rad/sample creates DC, 2w1, 2w2 and w1+w2.
for w_new in [0.0, 2 * 0.3, 2 * 1.1, 0.3 + 1.1]:
    ax.axvline(w_new / (2 * np.pi), color="r", ls=":", lw=0.9)
ax.set(xlabel="normalised frequency", ylim=(1e-2, 1e4))
ax.legend(); ax.set_title("A non-linear system invents frequencies (red lines: DC, 2ω₁, 2ω₂, ω₁+ω₂)")

---
## 15. The convolution theorem

Because complex exponentials pass through unchanged in shape, working in the frequency domain
turns convolution into ordinary multiplication:

$$y = x * h \quad\Longleftrightarrow\quad Y(e^{j\omega}) = X(e^{j\omega})\,H(e^{j\omega})$$

This is both a proof technique (many identities become one line) and — for long enough
kernels — a faster algorithm. First check that it is actually the same answer.

In [ ]:
rng = np.random.default_rng(5)
x = rng.normal(size=1000)
h = rng.normal(size=200)

L = len(x) + len(h) - 1
nfft = 1 << (L - 1).bit_length()        # next power of two

direct = np.convolve(x, h)
via_fft = np.fft.irfft(np.fft.rfft(x, nfft) * np.fft.rfft(h, nfft), nfft)[:L]

print("max difference:", np.max(np.abs(direct - via_fft)))
print("agree:", np.allclose(direct, via_fft))

### "Faster" deserves a measurement, not an assertion

Direct convolution costs $O(NM)$; the FFT route costs $O(L\log L)$ regardless of $M$. So the
FFT wins for **long kernels** — but it has real fixed overhead (three transforms and a
padding to a convenient length), and for a short kernel the direct method is simply better.

Find the crossover rather than guessing at it.

In [ ]:
import timeit

x_big = rng.normal(size=100_000)
print(f"{'len(h)':>8} {'direct [ms]':>13} {'fftconvolve [ms]':>18} {'winner':>9}")
print("-" * 52)
for M in [4, 16, 64, 256, 1024, 4096]:
    h_m = rng.normal(size=M)
    t_dir = min(timeit.repeat(lambda: np.convolve(x_big, h_m), number=1, repeat=3))
    t_fft = min(timeit.repeat(lambda: signal.fftconvolve(x_big, h_m), number=1, repeat=3))
    winner = "direct" if t_dir < t_fft else "FFT"
    print(f"{M:>8} {t_dir * 1e3:>13.2f} {t_fft * 1e3:>18.2f} {winner:>9}")

print()
print("scipy.signal.choose_conv_method picks for you:")
for M in [4, 64, 1024]:
    print(f"  len(h) = {M:>5}: {signal.choose_conv_method(x_big, rng.normal(size=M))}")

### And the dual: multiplication in time is convolution in frequency

The theorem runs both ways. Multiplying a signal by a window *convolves* its spectrum with the
window's spectrum — which is precisely the mechanism behind spectral leakage from notebook 01.

In [ ]:
N = 256
n = np.arange(N)
tone = np.cos(2 * np.pi * 40 / N * n)          # exactly bin 40
win = np.hanning(N)

f = np.arange(N // 2 + 1)
spec_tone = np.abs(np.fft.rfft(tone))
spec_win = np.abs(np.fft.rfft(win))
spec_prod = np.abs(np.fft.rfft(tone * win))

fig, ax = plt.subplots(1, 3, figsize=(14, 3))
ax[0].plot(f, spec_tone); ax[0].set(xlim=(30, 50), title="spectrum of the tone (one line)")
ax[1].plot(f, spec_win); ax[1].set(xlim=(0, 10), title="spectrum of the Hann window")
ax[2].plot(f, spec_prod); ax[2].set(xlim=(30, 50),
                                   title="product spectrum = tone ⊛ window")
for a_ in ax:
    a_.set_xlabel("bin")
fig.tight_layout()

print("The single spectral line has been smeared into the shape of the window's spectrum.")
print("That smearing IS spectral leakage -- convolution in the frequency domain.")

---
## 16. Circular convolution and the zero-padding trap

The DFT does not implement linear convolution. It implements **circular** convolution of
length $N$, where indices wrap around modulo $N$. Multiply two $N$-point DFTs and the tail
that should have extended past sample $N-1$ folds back onto the beginning.

The fix is zero-padding both signals to at least $N+M-1$ before transforming. This is not an
optimisation — it is a correctness requirement, and it is the mistake that produces
mysterious glitches at the start of an otherwise-correct filtered signal.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
h = np.array([1.0, 1.0, 1.0])

linear = np.convolve(x, h)                                        # length 6
N = len(x)
circular = np.fft.irfft(np.fft.rfft(x, N) * np.fft.rfft(h, N), N)  # length 4, WRONG

L = len(x) + len(h) - 1
padded = np.fft.irfft(np.fft.rfft(x, L) * np.fft.rfft(h, L), L)    # length 6, correct

print("linear convolution        :", linear)
print("N-point DFT (circular)    :", circular.round(6))
print("zero-padded to L=6        :", padded.round(6))
print()
print("The wrapped tail:", linear[N:], "got added back onto the first samples:")
print(f"  {linear[:2]} + {linear[N:]} = {linear[:2] + linear[N:]}   <- matches circular[:2]")

In [ ]:
# The same failure on a realistic signal, so you know what it looks like.
rng = np.random.default_rng(6)
x_long = np.cumsum(rng.normal(size=256)) * 0.1
h_long = signal.firwin(64, 0.15)

correct = signal.fftconvolve(x_long, h_long)[:256]
wrapped = np.fft.irfft(np.fft.rfft(x_long, 256) * np.fft.rfft(h_long, 256), 256)

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(correct, lw=1.5, label="correct (zero-padded)")
ax.plot(wrapped, "--", lw=1.5, label="circular (not padded)")
ax.axvspan(0, len(h_long), color="tab:red", alpha=0.12)
ax.text(len(h_long) / 2, ax.get_ylim()[1] * 0.8, "corrupted region",
        ha="center", fontsize=9, color="tab:red")
ax.legend(fontsize=8); ax.set_xlabel("n")
ax.set_title("Time-domain aliasing corrupts exactly the first M−1 samples")

print(f"max error overall           : {np.max(np.abs(correct - wrapped)):.4f}")
print(f"max error after sample {len(h_long)}   : {np.max(np.abs(correct[len(h_long):] - wrapped[len(h_long):])):.2e}")

> **The corruption is confined to the first $M-1$ samples.** That is what makes it so easy to
> miss: the bulk of your signal looks perfect, and only the leading edge is wrong. If you are
> processing a long recording in blocks and see a click at every block boundary, this is
> almost certainly why.

---
## 17. Block convolution: overlap-add

For a stream too long to hold in memory (or arriving in real time), you convolve block by
block. Because convolution is linear and distributive over the block decomposition, the pieces
just add — with each block's tail overlapping the next block's head.

`scipy.signal.oaconvolve` does this for you, but implementing it once makes the idea stick.

In [ ]:
def overlap_add(x, h, block=64):
    """Convolve x with h by processing x in blocks and summing overlapping tails."""
    M = len(h)
    y = np.zeros(len(x) + M - 1)
    for start in range(0, len(x), block):
        chunk = x[start:start + block]
        piece = np.convolve(chunk, h)                 # length len(chunk) + M - 1
        y[start:start + len(piece)] += piece          # the += is the "add"
    return y

rng = np.random.default_rng(7)
x = rng.normal(size=500)
h = signal.firwin(33, 0.25)

reference = np.convolve(x, h)
for block in [16, 64, 128]:
    out = overlap_add(x, h, block)
    print(f"block = {block:>4}: max error {np.max(np.abs(out - reference)):.2e}")

print()
print("scipy.signal.oaconvolve:", np.allclose(signal.oaconvolve(x, h), reference))

In [ ]:
# Visualise why it works: each block's output extends M-1 samples past its input.
x_demo = np.zeros(160); x_demo[::40] = 1.0        # impulses at block boundaries
h_demo = np.exp(-np.arange(25) / 6)
block = 40

fig, ax = plt.subplots(figsize=(11, 3.4))
total = np.zeros(len(x_demo) + len(h_demo) - 1)
for i, start in enumerate(range(0, len(x_demo), block)):
    chunk = np.zeros(len(x_demo)); chunk[start:start + block] = x_demo[start:start + block]
    piece = np.convolve(chunk[start:start + block], h_demo)
    padded = np.zeros_like(total)
    padded[start:start + len(piece)] = piece
    total += padded
    ax.plot(padded, lw=1.2, alpha=0.8, label=f"block {i} output")
ax.plot(total, "k", lw=2, label="sum = full convolution")
ax.legend(fontsize=8, ncol=5); ax.set_xlabel("n")
ax.set_title("Overlap-add: block outputs run past their block, and the overlaps sum")

print("matches direct convolution:", np.allclose(total, np.convolve(x_demo, h_demo)))

---
## 18. Deconvolution, and why it is hard

If $y = x * h$, can you recover $x$ from $y$ and $h$? In principle yes — divide in the
frequency domain:

$$X(e^{j\omega}) = \frac{Y(e^{j\omega})}{H(e^{j\omega})}$$

In practice this is one of the classic ill-posed problems. Wherever $H$ is small, dividing by
it amplifies whatever is there — and what is there is noise.

In [ ]:
# A clean echo: y[n] = x[n] + 0.6 x[n-D]. Its impulse response is easy to invert exactly.
fs = 8000
D = 400
n = np.arange(4000)
x_orig = signal.chirp(n / fs, f0=200, f1=1200, t1=0.5) * np.exp(-2 * n / fs)

h_echo = np.zeros(D + 1); h_echo[0] = 1.0; h_echo[D] = 0.6
y_echo = np.convolve(x_orig, h_echo)[:len(n)]

# The inverse system: y[n] - 0.6 * x_hat[n-D], run as an IIR filter.
x_recovered = signal.lfilter([1.0], h_echo, y_echo)

print(f"noise-free recovery error: {np.max(np.abs(x_recovered - x_orig)):.2e}")

fig, ax = plt.subplots(3, 1, figsize=(10, 5.5), sharex=True)
ax[0].plot(x_orig, lw=0.5); ax[0].set_title("original x[n]")
ax[1].plot(y_echo, lw=0.5, color="tab:orange"); ax[1].set_title("with echo, y[n]")
ax[2].plot(x_recovered, lw=0.5, color="tab:green")
ax[2].set_title("recovered by inverse filtering -- exact"); ax[2].set_xlabel("n")
fig.tight_layout()

In [ ]:
# Now add a tiny amount of noise. The same inverse filter falls apart.
rng = np.random.default_rng(8)
for snr_db in [60, 40, 20]:
    noise_power = np.mean(y_echo ** 2) / 10 ** (snr_db / 10)
    y_noisy = y_echo + rng.normal(0, np.sqrt(noise_power), len(y_echo))
    x_hat = signal.lfilter([1.0], h_echo, y_noisy)
    err = np.sqrt(np.mean((x_hat - x_orig) ** 2)) / np.sqrt(np.mean(x_orig ** 2))
    print(f"input SNR {snr_db:>3} dB  ->  relative recovery error {err:>8.3f}")

> **Notice how fast it degrades.** At 60 dB SNR — noise a million times weaker in power than
> the signal — recovery is already imperfect. This echo system has zeros close to the unit
> circle, so its inverse has poles close to the unit circle, and those nearly-unstable poles
> amplify noise enormously.

### Regularised (pseudo-)inverse

The standard fix is to stop dividing when $|H|$ gets small. Adding a small constant
$\lambda$ to the denominator gives the Wiener-style estimate

$$\hat{X} = \frac{H^*\,Y}{|H|^2 + \lambda}$$

which trades a little bias for a large reduction in noise amplification.

In [ ]:
def regularised_deconv(y, h, lam):
    """Frequency-domain deconvolution with Tikhonov regularisation."""
    L = len(y)
    Y = np.fft.rfft(y, L)
    H = np.fft.rfft(h, L)
    X = np.conj(H) * Y / (np.abs(H) ** 2 + lam)
    return np.fft.irfft(X, L)

noise_power = np.mean(y_echo ** 2) / 10 ** (30 / 10)     # 30 dB SNR
y_noisy = y_echo + rng.normal(0, np.sqrt(noise_power), len(y_echo))

naive = signal.lfilter([1.0], h_echo, y_noisy)

print(f"{'lambda':>10} {'relative error':>16}")
print("-" * 28)
print(f"{'naive':>10} {np.sqrt(np.mean((naive - x_orig)**2)) / np.sqrt(np.mean(x_orig**2)):>16.4f}")
best = None
for lam in [1e-6, 1e-4, 1e-2, 1e-1, 1.0]:
    est = regularised_deconv(y_noisy, h_echo, lam)[:len(x_orig)]
    rel = np.sqrt(np.mean((est - x_orig) ** 2)) / np.sqrt(np.mean(x_orig ** 2))
    print(f"{lam:>10.0e} {rel:>16.4f}")
    if best is None or rel < best[1]:
        best = (lam, rel, est)

print(f"\nbest lambda = {best[0]:.0e}")

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(x_orig, "0.6", lw=1.2, label="true x[n]")
ax.plot(naive, lw=0.7, alpha=0.8, label="naive inverse (noise-dominated)")
ax.plot(best[2], lw=1.2, label=f"regularised, λ={best[0]:.0e}")
ax.set_xlim(0, 1200); ax.legend(fontsize=8); ax.set_xlabel("n")
ax.set_title("Regularisation buys back a usable estimate")

> **The general lesson.** Convolution destroys information wherever $H(e^{j\omega}) \approx 0$
> — those frequency components are gone, and no algorithm recovers them. Deconvolution is
> therefore always a compromise between fidelity and noise amplification. This is the same
> mathematics behind image deblurring, seismic inversion, and CT reconstruction.

---
## 19. Applications: smoothing, differencing, reverb, images

Four uses of the same operation, to show how much of practical signal processing is "choose
the right $h$".

In [ ]:
# (a) Smoothing: different kernels, different frequency responses.
n = np.arange(300)
rng = np.random.default_rng(9)
clean = np.sin(2 * np.pi * 0.01 * n) + 0.5 * np.sin(2 * np.pi * 0.03 * n)
noisy = clean + rng.normal(0, 0.35, len(n))

tri = np.convolve(np.ones(11), np.ones(11))     # boxcar * boxcar = triangle, length 21
gauss = signal.windows.gaussian(21, 4)

kernels = {
    "boxcar (21)":    np.ones(21) / 21,
    "triangle (21)":  tri / tri.sum(),
    "gaussian (σ=4)": gauss / gauss.sum(),
}

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
ax[0].plot(n, noisy, "0.85", lw=0.7, label="noisy")
ax[0].plot(n, clean, "k--", lw=1, label="clean")
for name, k in kernels.items():
    ax[0].plot(n, np.convolve(noisy, k, mode="same"), lw=1.3, label=name)
ax[0].legend(fontsize=7, ncol=2); ax[0].set_xlabel("n"); ax[0].set_title("smoothing kernels")

for name, k in kernels.items():
    w, H = signal.freqz(k, worN=1024)
    ax[1].plot(w / np.pi, 20 * np.log10(np.abs(H) + 1e-12), label=name)
ax[1].set(xlabel="normalised frequency", ylabel="|H| [dB]", ylim=(-70, 5))
ax[1].legend(fontsize=7); ax[1].set_title("their frequency responses")
fig.tight_layout()

> **The boxcar's sidelobes are the point.** It looks like the simplest smoother, but its
> frequency response has large ripples that *pass* certain high frequencies almost unattenuated.
> The Gaussian kernel has no such sidelobes. This is the same main-lobe/sidelobe trade-off as
> the window functions in notebook 03 — because it is literally the same mathematics.

In [ ]:
# (b) Differencing: convolution as a derivative estimator.
t = np.linspace(0, 2, 400)
x_smooth = np.sin(2 * np.pi * 1.5 * t)
dt = t[1] - t[0]

first_diff = np.convolve(x_smooth, np.array([1.0, -1.0]) / dt, mode="same")
central = np.convolve(x_smooth, np.array([1.0, 0.0, -1.0]) / (2 * dt), mode="same")
exact = 2 * np.pi * 1.5 * np.cos(2 * np.pi * 1.5 * t)

fig, ax = plt.subplots(1, 2, figsize=(13, 3))
ax[0].plot(t, exact, "k", lw=2, label="exact derivative")
ax[0].plot(t, first_diff, lw=1, label="forward difference [1, −1]")
ax[0].plot(t, central, "--", lw=1.2, label="central difference [1, 0, −1]/2")
ax[0].set(xlim=(0.2, 1.8), xlabel="t [s]"); ax[0].legend(fontsize=8)
ax[0].set_title("differentiation is a convolution")

# Why differencing is dangerous on noisy data: it is a high-pass filter.
for k, name in [(np.array([1.0, -1.0]), "[1, −1]"), (np.array([1.0, 0.0, -1.0]) / 2, "[1,0,−1]/2")]:
    w, H = signal.freqz(k, worN=1024)
    ax[1].plot(w / np.pi, np.abs(H), label=name)
ax[1].set(xlabel="normalised frequency", ylabel="|H|")
ax[1].legend(fontsize=8); ax[1].set_title("gain rises with frequency -> amplifies noise")
fig.tight_layout()

print(f"forward-difference RMS error : {np.sqrt(np.mean((first_diff[5:-5] - exact[5:-5])**2)):.4f}")
print(f"central-difference RMS error : {np.sqrt(np.mean((central[5:-5] - exact[5:-5])**2)):.4f}")

In [ ]:
# (c) Reverb: convolve a dry signal with a room impulse response.
fs = 16000
rir_len = int(0.8 * fs)
rng = np.random.default_rng(10)

# Synthetic RIR: a direct pulse, a few early reflections, then exponentially decaying diffuse tail.
rir = np.zeros(rir_len)
rir[0] = 1.0
for delay_ms, amp in [(11, 0.55), (19, -0.4), (27, 0.32), (41, -0.25), (58, 0.18)]:
    rir[int(delay_ms * fs / 1000)] += amp
tail = rng.normal(size=rir_len) * np.exp(-np.arange(rir_len) / (0.18 * fs))
rir += 0.35 * tail
rir /= np.max(np.abs(rir))

# A dry signal: two short plucks.
dry = np.zeros(int(1.2 * fs))
for onset in [0.05, 0.55]:
    i0 = int(onset * fs)
    env = np.exp(-np.arange(int(0.12 * fs)) / (0.02 * fs))
    tone = np.sin(2 * np.pi * 440 * np.arange(len(env)) / fs)
    dry[i0:i0 + len(env)] += env * tone

wet = signal.fftconvolve(dry, rir)[:len(dry) + rir_len]

fig, ax = plt.subplots(3, 1, figsize=(10, 5.5))
ax[0].plot(np.arange(rir_len) / fs, rir, lw=0.5)
ax[0].set(title="room impulse response", xlabel="t [s]")
ax[1].plot(np.arange(len(dry)) / fs, dry, lw=0.5)
ax[1].set(title="dry signal", xlabel="t [s]")
ax[2].plot(np.arange(len(wet)) / fs, wet, lw=0.5, color="tab:green")
ax[2].set(title="convolved with the RIR -- reverberant", xlabel="t [s]")
fig.tight_layout()

print(f"dry length {len(dry)/fs:.2f} s + RIR {rir_len/fs:.2f} s -> wet {len(wet)/fs:.2f} s")
print("This is exactly how convolution reverb plugins work.")

In [ ]:
# Listen to the difference (audio widgets render in Jupyter).
from IPython.display import Audio, display
print("dry:")
display(Audio(dry, rate=fs, normalize=True))
print("wet:")
display(Audio(wet, rate=fs, normalize=True))

### (d) Two dimensions: the same operation on images

Convolution is not limited to one dimension. A 2-D kernel slides over an image in exactly the
same flip-and-slide manner. Every classical image filter — blur, sharpen, edge detection — is
a choice of kernel, and the early layers of a convolutional neural network learn these kernels
rather than having them specified.

In [ ]:
# Build a synthetic test image with edges, a gradient, and fine texture.
yy, xx = np.mgrid[0:120, 0:120]
img = np.zeros((120, 120))
img[20:60, 20:60] = 1.0                                  # a square
img[70:100, 30:90] = 0.6                                 # a rectangle
img += 0.35 * (xx / 120)                                 # a gradient
img[::7, :] += 0.15                                      # horizontal texture lines
img = np.clip(img, 0, 1.6)

kernels_2d = {
    "identity": np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], float),
    "box blur 5×5": np.ones((5, 5)) / 25,
    "gaussian blur": np.outer(*[signal.windows.gaussian(9, 2)] * 2) / signal.windows.gaussian(9, 2).sum() ** 2,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
    "Sobel ∂/∂x": np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]], float),
    "Laplacian": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float),
}

fig, ax = plt.subplots(2, 3, figsize=(12, 7))
for a_, (name, k) in zip(ax.ravel(), kernels_2d.items()):
    out = signal.convolve2d(img, k, mode="same", boundary="symm")
    a_.imshow(out, cmap="gray")
    a_.set_title(f"{name}   {k.shape}", fontsize=10)
    a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)
fig.tight_layout()

print("boundary='symm' reflects at the edges -- the 2-D version of the padding")
print("question from section 8.")

> **Separability, worth knowing.** A 2-D Gaussian blur equals a 1-D blur along rows followed by
> a 1-D blur along columns. For a $k \times k$ kernel that turns $O(k^2)$ work per pixel into
> $O(2k)$ — the reason large blurs are feasible at all.

In [ ]:
g1 = signal.windows.gaussian(15, 3); g1 /= g1.sum()

full_2d = signal.convolve2d(img, np.outer(g1, g1), mode="same", boundary="symm")
separated = signal.convolve2d(
    signal.convolve2d(img, g1[None, :], mode="same", boundary="symm"),
    g1[:, None], mode="same", boundary="symm")

print("separable result matches full 2-D:", np.allclose(full_2d, separated, atol=1e-12))
print(f"operations per pixel: {15*15} (full) vs {2*15} (separated)")

---
## 20. Common mistakes

Collected from the failure modes demonstrated above.

| # | Mistake | Consequence |
|---|---------|-------------|
| 1 | Assuming a system is LTI without checking | every prediction is silently wrong (§6) |
| 2 | Thinking linearity alone is enough | `n*x[n]` is linear but has no impulse response (§3) |
| 3 | Using `np.roll` for a delay | wraps around instead of zero-padding (§3) |
| 4 | Forgetting `dt` in continuous convolution | answer off by the sampling rate (§11) |
| 5 | Multiplying DFTs without zero-padding | circular convolution corrupts the first $M-1$ samples (§16) |
| 6 | Reading "h[n] → 0" as stability | need $\sum\lvert h\rvert < \infty$; `1/(n+1)` fails (§12) |
| 7 | Ignoring edge effects from zero padding | filtered signal dips at both ends (§8) |
| 8 | Naive deconvolution of noisy data | noise amplified without bound (§18) |
| 9 | Differencing noisy data directly | differencing is a high-pass; smooth first (§19) |
| 10 | Expecting an LTI filter to remove a harmonic it created | LTI systems cannot create frequencies, so the harmonic came from something non-linear (§14) |

### The one-paragraph summary

An LTI system is fully described by its impulse response $h$. The output for any input is
$y = x * h$, which in the frequency domain is $Y = XH$. Causality is $h[n]=0$ for $n<0$;
stability is $\sum|h[n]| < \infty$; cascading convolves impulse responses and multiplies
frequency responses. Everything else in the course is machinery for computing with these
facts more conveniently.

---
## 21. Exercises

**1. Classify these systems.** For each, determine linearity and time invariance using the
numerical tests from §2–§3, then prove your answer algebraically:
(a) $y[n] = x[n] - x[n-1]$ &nbsp; (b) $y[n] = x[2n]$ &nbsp; (c) $y[n] = \lvert x[n]\rvert$
&nbsp; (d) $y[n] = \sum_{k=0}^{n} x[k]$ &nbsp; (e) $y[n] = x[-n]$ &nbsp; (f) $y[n] = \cos(x[n])$.
For (b) and (e) think carefully about the time-invariance test — you will need to justify how
you compare.

**2. Hand convolution, then verify.** Convolve $x = [1, -2, 3]$ with $h = [2, 1, -1, 3]$ by
hand using the overlap table. Then check with `np.convolve`. Do the same for
$x = [1,1,1,1]$ with itself and explain the shape you get.

**3. Reconstruct the black box.** Given only that `black_box` is LTI, determine from its
impulse response: (a) is it FIR or IIR? (b) is it stable? Justify with $\sum|h|$. (c) is it
causal? (d) find a difference equation that reproduces it — hint: `scipy.signal.prony` or fit
the ratio of polynomials from the frequency response.

**4. The `1/(n+1)` system.** Convolve a bounded input with $h[n] = 1/(n+1)$ for increasing
signal lengths and plot the peak output against length. Show it grows like $\ln N$. Then find
a specific bounded input that maximises the output at $n=N$ (hint: what input aligns perfectly
with the flipped kernel?).

**5. Zero-padding forensics.** Take a 512-sample signal and a 128-tap filter. Compute the
product of their 512-point DFTs, invert, and compare against the correct linear convolution.
Plot the error and confirm it is confined to the first 127 samples. Then show that padding to
639 removes it entirely.

**6. Overlap-save.** §17 implemented overlap-**add**. Implement overlap-**save** (the variant
that discards corrupted samples from each block instead of summing tails) and verify it against
`np.convolve`. Which uses less memory, and which is easier to get right?

**7. Step-response identification with noise.** Add Gaussian noise to the measured step
response of `black_box`, then recover $h$ by differencing. Show how the recovered $h$ degrades
with noise level, and implement a better estimator (smooth before differencing, or fit a
low-order model). Quantify the improvement.

**8. Cascade equivalence, quantified.** Build a cascade of a 6th-order Butterworth low-pass and
a 6th-order high-pass. Verify the impulse-response and frequency-response identities. Then
implement both orderings in `float32` on a signal with large dynamic range and measure whether
the outputs still agree — connecting §10's warning to a number.

**9. Deconvolve a blurred image.** Blur the §19 test image with a known Gaussian kernel, add a
small amount of noise, and attempt to recover it with (a) naive frequency-domain division and
(b) the regularised estimator from §18. Sweep $\lambda$ and plot recovery error against it.

**10. Prove associativity by hand.** For $x=[1,2]$, $h_1=[1,1]$, $h_2=[1,-1]$, compute
$(x*h_1)*h_2$ and $x*(h_1*h_2)$ term by term on paper. Then explain, using the polynomial
interpretation from §7, why associativity is obvious.

**11. Build a convolution reverb.** Record or synthesise a room impulse response, then apply it
to a dry signal using `oaconvolve` in blocks small enough to run in real time at 48 kHz.
Measure the maximum block size your machine can sustain, and compare the cost against direct
convolution.

**12. Find the non-linearity.** Generate a signal, pass it through an unknown system that is
LTI-plus-a-small-cubic-term, and design a test that detects the non-linearity. Then estimate
its size. (Hint: feed two tones and look for intermodulation products at $2f_1 \pm f_2$.)

---

### Quick reference

| Concept | Statement / call |
|---------|------------------|
| linearity | $T\{ax_1+bx_2\} = aT\{x_1\}+bT\{x_2\}$ |
| time invariance | $T\{x[n-d]\} = y[n-d]$ |
| impulse response | `h = system(signal.unit_impulse(N))` |
| convolution sum | $y[n]=\sum_k x[k]h[n-k]$ → `np.convolve(x, h)` |
| output length | $N+M-1$ |
| identity | $x * \delta = x$ |
| delay | $x * \delta[n-d] = x[n-d]$ |
| accumulator | $x * u = $ `np.cumsum(x)` |
| cascade | $h_1 * h_2$ (order irrelevant) |
| parallel | $h_1 + h_2$ |
| continuous | `np.convolve(x, h) * dt` |
| causal | $h[n]=0$ for $n<0$ |
| BIBO stable | `np.sum(np.abs(h)) < inf` |
| step ↔ impulse | `h = np.diff(s, prepend=0)`, `s = np.cumsum(h)` |
| eigenfunction | $T\{e^{j\omega n}\} = H(e^{j\omega})e^{j\omega n}$ |
| convolution theorem | $Y = XH$ → `signal.fftconvolve(x, h)` |
| linear via DFT | pad both to $\ge N+M-1$ first |
| block processing | `signal.oaconvolve(x, h)` |
| 2-D | `signal.convolve2d(img, k, mode="same", boundary="symm")` |

### Where this leads

- **Notebook 03 §4–§10** — the same systems expressed as transfer functions, so that
  convolution becomes polynomial algebra and filter design becomes pole placement.
- **Notebook 03 §11–§12** — windows and spectral estimation, which are §15's dual theorem
  applied in earnest.
- The Laplace and z-transforms, whose entire purpose is to make $x * h$ into $X \cdot H$.